GPU: NVIDIA GeForce RTX 5070


In [ ]:
# Download the dataset
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")+'/Data/genres_original' 
# which is from https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification (YOU DONT NEED TO DOWNLOAD IT FROM HERE)

# Figure out file locations
dataset_dir = Path(path)
all_files = list(dataset_dir.glob("*/*.wav")) 
# all_files = [str(f.relative_to(dataset_dir)) for f in all_files] 



In [ ]:
# simple kagglehub download call (keeps original behaviour)
# replace with your actual dataset id/path if needed
dataset_path = kagglehub.dataset_download("and...", force=False)  # adjust args as your notebook originally did

# If kagglehub returns a path, make sure you point file_list to your wav files.
# For the purposes of this pipeline we assume audio files are under `dataset_path` or current dir.
root = Path(dataset_path) if dataset_path else Path('.')

# Collect a list of .wav files (recursively)
file_list = sorted([p for p in root.rglob('*.wav')])

# PyTorch Dataset for lazy audio loading and preprocessing
class AudioFilesDataset(Dataset):
    def __init__(self, file_list, transform=None, sample_rate=SAMPLE_RATE, duration=DURATION):
        self.files = file_list
        self.transform = transform
        self.sample_rate = sample_rate
        self.num_samples = int(sample_rate * duration)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        waveform, sr = torchaudio.load(path)  # waveform: (channels, time)
        # convert to mono if needed
        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # resample if sample rate doesn't match
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)
        # trim or pad to fixed length (duration)
        if waveform.size(1) > self.num_samples:
            waveform = waveform[:, :self.num_samples]
        elif waveform.size(1) < self.num_samples:
            pad = self.num_samples - waveform.size(1)
            waveform = F.pad(waveform, (0, pad))
        if self.transform is not None:
            waveform = self.transform(waveform)
        return waveform, path  # return path as a simple id

# Example transform: no-op (you can replace with STFT, mel, augmentations...)
def identity_transform(x):
    return x

dataset = AudioFilesDataset(file_list, transform=identity_transform)

# DataLoader (streams data lazily, with multiple workers)
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2
)

In [ ]:
# Download the dataset
# which is from https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification (YOU DONT NEED TO DOWNLOAD IT FROM HERE)
# Figure out file locations
# Create dataset
    # audio_tensor = tfio.audio.AudioIOTensor(file_path) # AAAAAAAAAAAAAAAAAAAAAFHDSLAFHUSDAGFIOSDBAYF
    # Read the file
    # Decode WAV (gives waveform tensor and sample rate)


def noisify(tensor, maxval=0.2):
    """Add uniform noise in [-maxval, maxval] to a torch waveform tensor (1, L)."""
    noise = (torch.rand_like(tensor) - 0.5) * 2.0 * maxval
    return tensor + noise

# Example playback helper using IPython.display.Audio (convert to numpy)
def display_waveform(waveform, rate=SAMPLE_RATE):
    arr = waveform.squeeze().cpu().numpy()
    display(Audio(arr, rate=rate))

In [ ]:
# Trying out the dataset
    # look at the tensor. look at it.
    # plot waveform
    # listen to  r e g g a e, relaaaax, rest your eyes, drink some water
    # You must suffer like I have.
# Take one batch from loader, create noisy versions and compute MSE loss (example)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for batch in loader:
    waveforms, paths = batch
    waveforms = waveforms.to(device)  # (B, 1, T)
    noisy_waveforms = noisify(waveforms, maxval=0.1).to(device)
    # simple MSE loss between clean and noisy as a toy example
    loss = F.mse_loss(waveforms, noisy_waveforms)
    print('example loss:', loss.item())
    break

In [ ]:
# noisify a batch
# take a batch and try it out
# Device and GPU info
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if torch.cuda.is_available():
    print('CUDA device name:', torch.cuda.get_device_name(0))

# Example UNet-like 1D model for denoising
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(out_ch, out_ch, 3, padding=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.block(x)

class UNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(1, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.pool = nn.MaxPool1d(2)
        self.dec3 = ConvBlock(128 + 64, 64)
        self.dec2 = ConvBlock(64 + 32, 32)
        self.out = nn.Conv1d(32, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d3 = F.interpolate(e3, scale_factor=2, mode='nearest')
        d3 = self.dec3(torch.cat([d3, e2], dim=1))
        d2 = F.interpolate(d3, scale_factor=2, mode='nearest')
        d2 = self.dec2(torch.cat([d2, e1], dim=1))
        return self.out(d2)

model = UNet1D().to(device)

# Example optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop skeleton (one epoch example)
model.train()
for epoch in range(1):
    for waveforms, _ in loader:
        waveforms = waveforms.to(device)
        noisy = noisify(waveforms, maxval=0.1).to(device)
        optimizer.zero_grad()
        pred = model(noisy)
        loss = loss_fn(pred, waveforms)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch} done, loss={loss.item()}')

In [ ]:
## Basic autoencoder
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))